# Integrated Intelligent Systems: Tutorial 3 - Body Motion Problem (Scientific Object) (3 Pts)

This notebook introduces a formulation body motion problem, central to designing and implementing Cognition-Enabled Robot Agents, as a description logic system.

## Part 1: Robot-World Manipulation System as Scientific Object (E1–E6) (1.5 Pt)

**Title:** Formalizing the Robot Agent: From Architecture to Description Logic Reasoners

![Alternative Text](RWMS.png)

### Scenario Background
An autonomous robotic agent (the PR2 in the EASE apartment) is deployed to execute a manipulation task ("Serve Eva a drink"). To validate this agent scientifically, we must formalize the system as a **Robot-World Manipulation System (RWMS)**, separate the controlling subsystem from the controlled body-world system, define trajectories as trees, and assess runs against an external problem standard using a description logic reasoner.

---

### Multi-Question Student Task Sheet (0.25 Pt x 6)

1. **E1 · Architectural Cut & Subsystem Separation**
   - In the graphical representation of the RWMS, a strict architectural "cut" is placed between the manipulation executive and the body-world system ($[B \bowtie W]$). Explain the nature of this cut: why is the manipulation executive classified as a `dul:Agent` (non-physical program) while the body-world system is classified under physical objects, and how do they interact exclusively across the boundary?
  
        The Manipulation Executive is a non physical (software) dul:Agent responsible for making decisions, while the Body-World System consists of physical objects that interact with the environment and physically carry out those decisions.
         Only action tokens and observations cross the boundary between the two subsystems. Action tokens are sent from the Manipulation Executive to the Body-World System, while observations consist of information perceived by the robot's sensors sent from the Body-World System back to the Manipulation Executive.

2. **E2 · Forward-Branching Time & World Choices**
   - During an operation on a fridge door or container seal, the same action token (e.g., a position-controlled pull) can lead to completely divergent physical outcomes depending on world choices (e.g., seal releases vs. seal holds tight). Draw or describe the structure of the resulting courses, and explain why the possible courses form a tree that branches *only forward* from the initial state $x_0$.
  
        All action courses begin at the same initial state x_0, but the same action token can lead to different physical outcomes depending on the state of the environment (world choices). For example, when the robot pulls the refrigerator door with the same force, the seal might release or remain closed.
         These different outcomes create possible courses that share the same past (the same initial state and action token), but that later can diverge into different futures. Once an outcome occurs, it becomes part of that course's history and can't be overwritten by future events. For example, if the robot failed to first open the door but now succeeds, that does not change the fact it first failed to open the door: that failure is part of the course's history.
         Therefore, the possible courses form a tree that branches only forward from the initial state.

3. **E3 · Trajectory Classification & Priority Hierarchies**
   - The problem standard $P$ sorts every physical trajectory into one of four classes: **SOL** (Solution), **REPD** (Repaired), **NON** (Nonachievement), and **PROH** (Prohibited). Consider two runs that both successfully place the drink within Eva's reach, but where Run A tears off the container handle (`P4` violation) while Run B maintains licensed contact. Classify both runs and explain why the classifier grades the *entire course* rather than just the final end-state.
  
        The problem standard P evaluates the entire trajectory rather than just the final outcome. Run B would be classified as SOL because the robot successfully delivers the drink while maintaining licensed contact and respecting the task constraints. Run A on the other hand, is classified as PROH because, despite delivering the drink, the robot tears of the container handle, which is a prohibited action, thus violating constraint P4.
         The classifier evaluates the complete course because achieving the goal does not override a prohibited action.

4. **E4 · Description Logic Partonomy & DUL Constraints**
   - When formalizing the RWMS in description logic under DOLCE Ultra Lite (DUL), we define an RWMS as having *exactly one* executive and *exactly one* body-world system (`RWMS = (=1 hasComponent.ManipExecutive) ⊓ (=1 hasComponent.BodyWorldSystem)`). Using the transitivity of `dul:isPartOf`, explain how a reasoner (such as HermiT) automatically entails that the PR2's head camera is a component of the RWMS without explicitly hardcoding every part-whole link.
  
        If the PR2's head camera is part of the PR2, the PR2 is part of the Body-World System and the Body-World System is part of the Robot-World Manipulation System, then a reasoner can infer that the head camera is part of the Robot-World Manipulation System. This is possible because the dul:isPartOf relationship is transitive, meaning the reasoner can infer part-whole relationships without explicitly defining every connection.
         Furthermore, the Robot-World Manipulation System must contain exactly one Manipulation Executive and one Body-World System according to its cardinality constraints (not more, not less).

5. **E5 · Ontological Inconsistency & Entailment Limits**
   - An inconsistency in description logic serves as a strict analytical tool. Formulate a competency question that results in an ontological inconsistency when attempting to classify the *manipulation problem itself* as a physical component inside the RWMS. Furthermore, explain why numerical physical laws (such as Coulomb friction $2 \cdot \mu \cdot F \ge m \cdot (g + a)$) must be handled outside description logic by the semantic digital twin's causal model.
  
        We could formulate the following competency question: Can the manipulation problem "Serve Eva a drink" be classified as a physical component of the Robot-World Manipulation System?"
     
     A manipulation problem is an abstract entity rather than a physical component. Since abstract entities and physical objects are defined as disjoint categories in the ontology, an object cannot be physical and abstract all at once. Therefore, classifying an individual as both would create an ontological inconsistency.
         Furthermore, numerical physical laws (like Coulomb's friction law) must be handled by the semantic digital twin's causal model, rather than by Description Logic. These laws require numerical calculations to determine, for example, whether the gripper applies enough force to prevent a milk carton from slipping. Description Logic can male logical inferences, but it cannot independantly perform these typical calculations or simulate their consequences.

6. **E6 · Strength vs. Truth in Agent Validation**
   - The design thesis dictates that an agent decides within its believed causality, which can be tested for logical *strength* via competency queries and for *truth* via physical experiments. Compare how an axiom set that lacks a friction law ($\text{K}_0$) performs on competency queries versus an axiomatization containing explicit dynamical laws ($\text{K}_1/\text{K}_2$), and state what kind of test is required to validate the actual physical truth of those beliefs.

     An agent makes decisions based on its believed causality. Its decisions can be tested for logical strength and physical truth. Without a friction law, K_0 cannot answer competency queries requiring friction calculations. On the other hand, K_1 and K_2 contain explicit dynamical laws, allowing them to derive stronger conclusions from its axioms.
        It is worth noting however, that logical strength does not guarantee physical truth. Even logically stronger models like K_1 and K_2 must be validated through physical experiments to determine whether their predictions correspond to reality (we need to physically test their predictions against the physical outcomes in order to validate their predictions).
         

## Part 2: Robot Motion Problem and Description Logics (1.5 Pt)

**Title:** Formulating and Querying the Robot Motion Problem with Knowledge Bases & Description Logics

![Alternative Text](PR2.png)

### Scenario Background
Using the interactive Jupyter notebook and RViz simulation environment (`bmp.rviz`, `setup_demo()`), the PR2 mobile manipulator interacts with the EASE apartment model (including the fridge and various kitchen drawers). Instead of hardcoding trajectories, the robotic executive queries a knowledge base (represented via Description Logics / Prolog reasoning) to link task requests to state changes, causes (motions), and capabilities.

[Go to the programming Lab](https://binder.intel4coro.de/v2/gh/maltehue/giskard-examples/bmp_vrb?urlpath=/lab/tree/notebooks/bodyMotionProblem.ipynb)
---

### Multi-Question Student Task Sheet (0.25 Pt x 6)

1. **E7 · Knowledge Base Grounding & Task Requests**
   - Execute the initial query `taskRequest(Request, StateChange).`Examine the resulting solution bindings (e.g., `{'Request': 'open_fridge', 'StateChange': 'open(fridge)'}` and various drawer operations). Explain how mapping abstract task requests to discrete state changes forms the top-level specification layer for the manipulation executive.
  
        The taskRequest(Request, StateChange) query maps abstract task requests to the desired changes in the environment. For example, the request open_fridge corresponds to the state change open(fridge).
         This forms the top-level specification layer for the Manipulation Executive because it defines what the robot must achieve, without specifying the physical motions required or even if the robot is capable of performing them.

2. **E8 · Extending Queries for Causal Motions & Parameters**
   - Extend the query to incorporate motion causes: `taskRequest(Request, StateChange), causes(Motion, StateChange, MotionParam).` Explain how the reasoner bridges the gap between a desired symbolic state change and continuous motion parameters (such as `envJointGoal`, target joint angles, and handle identifiers).
  
        The causes relation identifies which specific motions can produce the desired state change specified by taskRequest. Therefore, it connects an abstract task request to a concrete motion specification.
         This relation provides two extra variables: Motion, which identifies the type of motion, and MotionParam which specifies the required parameters.
         For example, for opening the refrigerator we get the following pairing: {'Request': 'open_fridge', 'StateChange': 'open(fridge)', 'Motion': 'envJointGoal', 'MotionParam': [0.8, b'fridge_door_joint', b'fridge_door_handle']}. This means that opening the refrigerator involves an envJointGoal motion with a target joint angle of 0.8 radians, the refrigerator door joint, and its handle.
         It is worth nothing though, that this does not establish whether our robot is physically able to perform this specifc motion to produce the desired state change in the environment.

3. **E9 · Integrating Robot Capabilities (`canPerform`)**
   - Finalize the query by adding the robot capability constraint: `taskRequest(Request, StateChange), causes(Motion, StateChange, MotionParam), canPerform(Robot, Motion, MotionParam).`Describe what role the `Robot` variable (`'pr2'`) plays in filtering feasible trajectories through the reasoner.
  
        The canPerform relation identifies which robot (if any) is capable of performing a specific motion with the required parameters to achieve the desired state change in the environment.
         For example, if the query returns Robot: pr2, it means that the knowledge base identifies the PR2 as capable of performing that motion. This helps the reasoner filter out motions that the available robots can't perform (but does not guarantee physical success when performing them).

4. **E10 · Specializing Request Scope (`open_fridge` vs. `open_anything`)**
   - Modify the query request parameter from general categories like `open_anything` to specifically target `open_fridge`. Explain how refining the scope in the knowledge base reduces the solution search tree and prevents unauthorized or unintended state changes in the kitchen environment.
  
     Using open_fridge instead of open_anything allows the knowledge base to filter out task requests that do not involve opening the refrigerator. As we observed in the programming lab, open_anything returns possible motions for opening several kitchen objects, including drawers and the refrigerator, while open_fridge specifically restricts the results to the refrigerator.
        This reduces the number of possible solutions the reasoner must consider, thus making the search more efficient. It also reduces the risk of unauthorized state changes, such as opening a drawer when the user only requested the refrigerator.

5. **E11 · Visualization and Simulation Linkage in RViz**
   - With the background shell command executing `rviz -d .../bmp.rviz` and displaying the PR2 interacting with the fridge handle, explain how the motion parameters derived from the reasoner's output (`[0.8, b'fridge_door_joint', b'fridge_door_handle']`) are executed as physical joint trajectories in the simulation.
  
     The knowledge base provides the motion goal that the robot's motion planning and control system must achieve. For example, the motion parameters `[0.8, fridge_door_joint, fridge_door_handle]` specify the target angle of the refrigerator door, the joint to be moved, and the handle involved.
     The motion planner and controller use this information to determine the robot's required joint trajectories and interactions with the environment. The simulation represents the resulting changes in the robot's and refrigerator's configurations, while RViz visualizes them so that we can inspect the movement.
        It is worth noting though, that the visualization does not guarantee that the motion will succeed physically.

6. **E12 · Ontological Limits & Missing Dynamics**
   - While the knowledge base successfully infers *which* motion achieves a state change, explain why a pure Description Logic or symbolic Prolog reasoner cannot verify whether a specific grasp force will cause a wet carton to slip, and identify which subsystem must handle those physical calculations.
  
     Symbolic reasoning can identify which motion can achieve a desired state change, but it cannot independently determine if a specific grip force will prevent a wet milk carton from slipping. This requires numerical calculations involving friction, mass, gravity, etc. These calculations must be handled by the semantic digital twin's causal model, using physical laws such as Coulomb friction, rather than by Description Logic or symbolic Prolog reasoning.

# Submission guidelines

- Deadline: 24.11.2026, 4 PM
- Submission as Git repo commits pushed within deadlines (If not yet, create a repo with the name **iis-assignment-win26** for the whole semester and invite me as collaborator)
- Put the textual answers in notebook and all related media (images, rules, etc) should then be put into a folder called **Assignment3**. 